# Lab 4.1 – Golden Dataset for DFD Evaluations

Inspect the ≥20-row golden set, validate schema, and run deterministic metrics on a sample of rows.

In [ ]:
# --- Environment bootstrap (Colab + local + Docker) ---
import os, sys
from pathlib import Path

def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

REPO_URL = "https://github.com/fischer3-net/accurate_secure_rag_systems.git"
LAB_DIR = "labs/04-evaluation"

if _in_colab():
    REPO_ROOT = Path("/content/accurate_secure_rag_systems")
    if not REPO_ROOT.exists():
        get_ipython().system(f"git clone --depth 1 {REPO_URL} {REPO_ROOT}")
    LAB = REPO_ROOT / LAB_DIR
    os.chdir(LAB)
    sys.path.insert(0, str(LAB))
    sys.path.insert(0, str(REPO_ROOT / "labs" / "01-chunking"))
    sys.path.insert(0, str(REPO_ROOT / "labs" / "03-skills"))
    get_ipython().run_line_magic("pip", "install -q pydantic python-dotenv langchain-text-splitters langchain-core pytest pyyaml pandas")
    print("Colab ready | LAB =", LAB)
else:
    LAB = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    if str(LAB) not in sys.path:
        sys.path.insert(0, str(LAB))
    print("Local ready | LAB =", LAB)


In [ ]:
import sys
from pathlib import Path
from collections import Counter

LAB = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(LAB))

from src.dataset import load_golden_dataset, validate_golden_dataset

GOLDEN = LAB / "data" / "golden_dataset.jsonl"
FIXTURES = LAB / "data" / "fixtures"

rows = load_golden_dataset(GOLDEN)
print(f"Loaded {len(rows)} golden rows")
print("Validation errors:", validate_golden_dataset(GOLDEN, FIXTURES))
print("Tags:", Counter(t for r in rows for t in r.tags))
print("Statuses:", Counter(r.expected_status for r in rows))

In [ ]:
for r in rows[:5]:
    print(f"{r.id}  [{r.expected_status:6}]  fixture={r.dfd_fixture}  controls={r.ground_truth_control_ids}")
    print(f"     Q: {r.question[:80]}")

In [ ]:
from src.eval_runner import run_evaluation

report = run_evaluation(
    golden_path=GOLDEN,
    fixtures_dir=FIXTURES,
    corpus_path=LAB / "data" / "rag_chunks.jsonl",
    min_mean_overall=0.50,
    min_mean_control_hit_rate=0.40,
)
print("mean_overall:", round(report["mean_overall"], 3))
print("mean_control_hit_rate:", round(report["mean_control_hit_rate"], 3))
print("mean_status_match:", round(report["mean_status_match"], 3))
print("thresholds_passed:", report["thresholds_passed"])
print("messages:", report["messages"])

## Coverage notes (for Capstone)

- Add rows for additional DFD shapes your organisation uses.
- Expand `ground_truth_contexts` when enabling Ragas Faithfulness.
- Keep deterministic gates strict; treat LLM-judge scores as advisory until calibrated.